In [0]:
# ==================================
# All Imports
# ==================================
import pandas as pd
import glob
import requests
import time
import numpy as np
from datetime import datetime
from pyspark.sql import SparkSession, DataFrame

# ==================================
# 1. Ingest Function (from ingest.py)
# ==================================
def ingest_sales_data(base_path_pattern: str) -> pd.DataFrame:
    """Reads all daily sales CSV files and merges them."""
    print("Starting data ingestion...")
    file_paths = glob.glob(base_path_pattern)
    if not file_paths:
        print("Warning: No files found.")
        return pd.DataFrame()
    sales_df = pd.concat((pd.read_csv(f) for f in file_paths), ignore_index=True)
    print(f"Ingestion complete. Total rows: {len(sales_df)}")
    return sales_df

# ==================================
# 2. Transform & Enrich Function (from transform.py and enrich.py)
# ==================================
def transform_and_enrich_data(df: pd.DataFrame) -> pd.DataFrame:
    """Applies all cleaning, feature engineering, and enrichment logic."""
    print("Starting data transformation and enrichment...")

    # --- Clean Sales Data ---
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    df.fillna(value={"unit_sales": 0}, inplace=True)
    df.dropna(subset=['dollar_sales'], inplace=True)
    df['date'] = pd.to_datetime(df['date'], format='mixed')
    df['unit_sales'] = df['unit_sales'].astype(int)
    df['promotion_flag'] = df['promotion_flag'].astype(bool)
    df['store_zip'] = df['store_zip'].str.replace('XX', '00')
    df['rev_per_unit'] = (df['dollar_sales'] / df['unit_sales']).round(2)
    df.replace([np.inf, -np.inf], 0, inplace=True)
    
    # --- Enrich with Weather Data ---
    API_KEY = "5883b9ac08253fbd080c073b1832a6a8"
    LAT, LON = 34.0522, -118.2437
    weather_data = []
    unique_dates = df['date'].dt.date.sort_values().unique()

    for date_obj in unique_dates:
        try:
            unix_timestamp = int(datetime.combine(date_obj, datetime.min.time()).timestamp())
            url = (f"https://api.openweathermap.org/data/3.0/onecall/timemachine"
                   f"?lat={LAT}&lon={LON}&dt={unix_timestamp}&appid={API_KEY}&units=imperial")
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            weather_data.append({"date": date_obj, "temp": data["data"][0]["temp"], "humidity": data["data"][0]["humidity"]})
        except requests.exceptions.RequestException as e:
            print(f"Warning: API call failed for {date_obj}: {e}")
        time.sleep(0.5)

    if weather_data:
        weather_df = pd.DataFrame(weather_data)
        weather_df['date'] = pd.to_datetime(weather_df['date'])
        df = df.merge(weather_df, how="left", on="date")
    
    print("Transformation and enrichment complete.")
    return df

# ==================================
# 3. Load Function (from load.py)
# ==================================
def load_data(df: pd.DataFrame, spark: SparkSession, table_name: str):
    """Saves the final DataFrame as a Delta table in Databricks."""
    print(f"Loading data into Delta table: {table_name}...")
    final_spark_df = spark.createDataFrame(df)
    (final_spark_df.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(table_name))
    print("Data loading complete.")

# ==================================
# 4. Main Orchestrator Function (from main.py)
# ==================================
def main():
    """Main function to run the complete ETL pipeline."""
    spark = SparkSession.builder.appName("RetailETLPipeline").getOrCreate()
    
    # --- Configuration ---
    SOURCE_PATH = "/Volumes/workspace/default/project_files/LA_Retail_Sales_By_Day/*.csv"
    FINAL_TABLE = "la_sales_pipeline"
    
    print("--- Starting ETL Pipeline ---")
    
    # --- Run ETL Steps ---
    raw_df = ingest_sales_data(base_path_pattern=SOURCE_PATH)
    if not raw_df.empty:
        transformed_df = transform_and_enrich_data(df=raw_df)
        load_data(df=transformed_df, spark=spark, table_name=FINAL_TABLE)
        print("\n--- ETL pipeline completed successfully! ---")
    else:
        print("Pipeline stopped: No data was ingested.")

# ==================================
# 5. Execute the Pipeline
# ==================================
if __name__ == "__main__":
    main()

--- Starting ETL Pipeline ---
Starting data ingestion...
Ingestion complete. Total rows: 750
Starting data transformation and enrichment...
Transformation and enrichment complete.
Loading data into Delta table: la_sales_pipeline...
Data loading complete.

--- ETL pipeline completed successfully! ---


--- Starting Extract Phase ---
Successfully extracted 750 rows into the 'sales_bronze' table.
